In [3]:
!pip install torchcam

In [5]:
import torch
import os
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from PIL import Image
from torchcam.methods import GradCAM
from tqdm import tqdm

# Nastavenie zariadenia
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Načítanie modelu (uprav podľa potreby)
model_path = "/home/jovyan/data/lightning/LiviaMurankova/new/namnozene_meteory/najlepsie_modely/najlepsie_modely/yolov11_vacsie_obrazky.pt"
model = torch.load(model_path, weights_only=False).to(device)
model.eval()

# Cesta k testovacím obrázkom a labelom
test_images_dir = "/home/jovyan/data/lightning/LiviaMurankova/new/namnozene_meteory/najlepsie_modely/vyhodnotenie/test/images"
test_labels_dir = "/home/jovyan/data/lightning/LiviaMurankova/new/namnozene_meteory/najlepsie_modely/vyhodnotenie/test_yolo/labels"

gradcam_dir = '/home/jovyan/data/lightning/LiviaMurankova/new/namnozene_meteory/ensemble_learning/gradcam'
os.makedirs(gradcam_dir, exist_ok=True)

# Transforms (uprav podľa potreby)
transform = transforms.Compose([
    transforms.Resize((1408, 1408)),
    transforms.ToTensor()
])

# Grad-CAM
def apply_grad_cam(image_tensor, model, target_layer):
    # Získaj instanciu Grad-CAM s cieľovou vrstvou
    cam = GradCAM(model, target_layer)
    # Vypočítať CAM
    activation_map = cam(image_tensor.unsqueeze(0))
    return activation_map

# Funkcia na vizualizáciu a ukladanie Grad-CAM obrázkov
def visualize_and_save(image, cam_image, img_name, alpha=0.5):
    fig, ax = plt.subplots(figsize=(10, 5))
    
    ax.imshow(transforms.ToPILImage()(image))
    ax.imshow(cam_image.squeeze().numpy(), cmap='jet', alpha=alpha)
    ax.axis('off')
    
    # Uloženie obrázku do priečinka 'gradcam'
    output_path = os.path.join(gradcam_dir, f'gradcam_{img_name}')
    plt.savefig(output_path)
    plt.close()

# Extrahovanie a vizualizácia
image_files = sorted([f for f in os.listdir(test_images_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])

for img_name in tqdm(image_files, desc="Processing Images"):
    img_path = os.path.join(test_images_dir, img_name)
    image = Image.open(img_path).convert('RGB')
    image_tensor = transform(image).to(device)
    
    # Vypočítať CAM
    cam_result = apply_grad_cam(image_tensor, model, 'features')  # Uprav 'features' na konkrétnu vrstvu modelu
    visualize_and_save(image_tensor, cam_result, img_name)

# Upozornenie: Uprav 'features' na názov vrstvy z tvojho modelu, ktorý chceš použiť pre Grad-CAM

AttributeError: 'dict' object has no attribute 'to'

In [16]:
model_path = "/home/jovyan/data/lightning/LiviaMurankova/new/namnozene_meteory/najlepsie_modely/najlepsie_modely/yolov11_vacsie_obrazky.pt"
image_path = "/home/jovyan/data/lightning/LiviaMurankova/new/namnozene_meteory/najlepsie_modely/vyhodnotenie/test/images/M20211101_034048_AMOS-CE_P.jpg"

import torch
from torchcam.methods import GradCAM
from torchcam.utils import overlay_mask
from PIL import Image
import matplotlib.pyplot as plt
from torchvision.transforms import Compose, Resize, ToTensor, Normalize, Lambda

# Assuming your YOLO model is correctly imported
from ultralytics import YOLO  # Adjust this import to your actual model's import

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the model
model = YOLO(model_path).to(device)
model.eval()

# Set up image transformations with proper normalization
transform = Compose([
    Resize((1408, 1408)),  # Resize the image to the size expected by the model
    ToTensor(),  # Convert the image to a tensor
    Lambda(lambda x: x / 255.0 if x.max() > 1.0 else x),  # Normalize if not already
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Standard normalization
])


# Hook function to capture features
features = None
def hook_fn(m, i, o):
    global features
    print("Hook executed")
    if o is not None:
        features = o
        print(f"Features shape: {features.shape}")
    else:
        print("No data passed through this layer")

# Traverse to the submodule where the hook should be registered
submodule = model
path_parts = ["model", "model", "8", "m", "0", "cv2", "conv"]
for part in path_parts:
    submodule = getattr(submodule, part)

# Register the hook
submodule.register_forward_hook(hook_fn)
print(f"Hook registered on {submodule}")

# Load and process the image
img = Image.open(image_path).convert('RGB')
input_tensor = transform(img).unsqueeze(0).to(device)

# Clear previous features
features = None

# Perform inference
print("Performing inference...")
output = model(input_tensor)

# Check if features were captured
if features is None:
    print("No features captured, check the layer name and hook setup")
else:
    print(f"Features captured successfully, shape: {features.shape}")

# Initialize Grad-CAM with the captured features if available
if features is not None:
    cam_extractor = GradCAM(model, target_layer=submodule)
    cam = cam_extractor(features.squeeze(0))
    cam_image = Image.fromarray(cam.cpu().numpy().squeeze(), mode='F')
    result = overlay_mask(Image.fromarray(img), cam_image, alpha=0.5)

    # Display the results
    plt.imshow(result)
    plt.axis('off')
    plt.show()
else:
    print("Failed to capture features for Grad-CAM.")


Hook registered on Conv2d(256, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
Performing inference...

WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 2.640000104904175. Dividing input by 255.
0: 1408x1408 (no detections), 58.8ms
Speed: 0.0ms preprocess, 58.8ms inference, 4.5ms postprocess per image at shape (1, 3, 1408, 1408)
No features captured, check the layer name and hook setup
Failed to capture features for Grad-CAM.


In [7]:
# Predpokladám, že model YOLOv11 je už načítaný do premennej `model`

# Vypísanie všetkých vrstiev modelu a ich názvov
for name, module in model.named_modules():
    print(name)



model
model.model
model.model.0
model.model.0.conv
model.model.0.bn
model.model.0.act
model.model.1
model.model.1.conv
model.model.1.bn
model.model.2
model.model.2.cv1
model.model.2.cv1.conv
model.model.2.cv1.bn
model.model.2.cv2
model.model.2.cv2.conv
model.model.2.cv2.bn
model.model.2.m
model.model.2.m.0
model.model.2.m.0.cv1
model.model.2.m.0.cv1.conv
model.model.2.m.0.cv1.bn
model.model.2.m.0.cv2
model.model.2.m.0.cv2.conv
model.model.2.m.0.cv2.bn
model.model.3
model.model.3.conv
model.model.3.bn
model.model.4
model.model.4.cv1
model.model.4.cv1.conv
model.model.4.cv1.bn
model.model.4.cv2
model.model.4.cv2.conv
model.model.4.cv2.bn
model.model.4.m
model.model.4.m.0
model.model.4.m.0.cv1
model.model.4.m.0.cv1.conv
model.model.4.m.0.cv1.bn
model.model.4.m.0.cv2
model.model.4.m.0.cv2.conv
model.model.4.m.0.cv2.bn
model.model.5
model.model.5.conv
model.model.5.bn
model.model.6
model.model.6.cv1
model.model.6.cv1.conv
model.model.6.cv1.bn
model.model.6.cv2
model.model.6.cv2.conv
model.